# 🧹 Datenbereinigung mit Python — Beispiel-Notebook

Dieses Notebook zeigt anhand eines **künstlich "verschmutzten" Datensatzes** typische Schritte der Datenbereinigung:

- Fehlende Werte
- Duplikate
- Tippfehler, Schreibweisen und Formate
- Wide- & Long-Format

Einfach Zelle für Zelle ausführen (Shift + Enter).

## Vorbereitung: Bibliotheken & Beispieldatensatz

Wir erzeugen einen kleinen, absichtlich unsauberen Datensatz, damit alle folgenden Befehle etwas zu tun haben.

In [1]:
# Bibliotheken importieren
import pandas as pd
import numpy as np

# Für Fuzzy Matching später (einmalig installieren, in Colab per !pip)
!pip install rapidfuzz -q

In [2]:
# Künstlich "unsauberen" Beispieldatensatz erzeugen, Normalerweise würdet ihr hier euren Datensatz laden
data = {
    'ID': [1, 2, 3, 4, 5, 6, 7, 8, 1, 9],  # ID 1 kommt doppelt vor -> Duplikat
    'Name': [' Anna ', 'anna', 'Bernd', 'bernd ', 'Chiara', 'CHIARA', 'Dénis',
              'Denis', ' Anna ', 'Elke'],
    'Stadt': ['München', 'muenchen', 'Köln', 'koeln', 'Berlin', 'berlin',
               'Hamburg', 'hamburg', 'München', np.nan],
    'PLZ': ['80331', '80331', '50667', '50667', '10115', '10115',
             '20095', '20095', '80331', '99999'],
    'Alter': ['29', '29', '41', 'x41', '35', '35', '52', np.nan, '29', '60'],
    'Einkommen_2023': [3200, 3200, 4100, 4100, np.nan, 2950, 3800, 3800, 3200, 5000],
    'Einkommen_2024': [3300, 3300, 4200, 4200, 3100, 3000, np.nan, 3900, 3300, 5200],
}

df = pd.DataFrame(data)
df

,ID,Name,Stadt,PLZ,Alter,Einkommen_2023,Einkommen_2024
0,1,Anna,München,80331,29,3200.0,3300.0
1,2,anna,muenchen,80331,29,3200.0,3300.0
2,3,Bernd,Köln,50667,41,4100.0,4200.0
3,4,bernd,koeln,50667,x41,4100.0,4200.0
4,5,Chiara,Berlin,10115,35,NaN,3100.0
5,6,CHIARA,berlin,10115,35,2950.0,3000.0
6,7,Dénis,Hamburg,20095,52,3800.0,NaN
7,8,Denis,hamburg,20095,NaN,3800.0,3900.0
8,1,Anna,München,80331,29,3200.0,3300.0
9,9,Elke,NaN,99999,60,5000.0,5200.0


## 1. Fehlende Werte
### 1.1 Fehlende Werte erkennen

df.isnull().sum()          # NaNs pro Spalte zählen

In [3]:
df.isnull().sum(axis=1)    # NaNs pro Zeile zählen

0    0
1    0
2    0
3    0
4    1
5    0
6    1
7    1
8    0
9    1
dtype: int64

df.isnull().any()          # Welche Spalten haben NaNs?

In [4]:
df.isnull().mean() * 100   # Anteil NaNs in %

ID                 0.0
Name               0.0
Stadt             10.0
PLZ                0.0
Alter             10.0
Einkommen_2023    10.0
Einkommen_2024    10.0
dtype: float64

In [5]:
df[df.isnull().any(axis=1)]  # Zeilen mit mind. einem NaN

,ID,Name,Stadt,PLZ,Alter,Einkommen_2023,Einkommen_2024
4,5,Chiara,Berlin,10115,35,NaN,3100.0
6,7,Dénis,Hamburg,20095,52,3800.0,NaN
7,8,Denis,hamburg,20095,NaN,3800.0,3900.0
9,9,Elke,NaN,99999,60,5000.0,5200.0


In [7]:
df.info()                  # Überblick inkl. Non-Null-Counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              10 non-null     int64  
 1   Name            10 non-null     object 
 2   Stadt           9 non-null      object 
 3   PLZ             10 non-null     object 
 4   Alter           9 non-null      object 
 5   Einkommen_2023  9 non-null      float64
 6   Einkommen_2024  9 non-null      float64
dtypes: float64(2), int64(1), object(4)
memory usage: 688.0+ bytes


### 1.2 Fehlende Werte behandeln

⚠️ Wichtig: `dropna()` und `fillna()` geben standardmäßig eine **Kopie** zurück. Wir arbeiten hier bewusst mit Kopien (`df_...`), damit `df` als Ausgangsbasis erhalten bleibt.

In [9]:
df_dropna_rows = df.dropna()              # Zeilen mit NaN löschen
df_dropna_rows

,ID,Name,Stadt,PLZ,Alter,Einkommen_2023,Einkommen_2024
0,1,Anna,München,80331,29,3200.0,3300.0
1,2,anna,muenchen,80331,29,3200.0,3300.0
2,3,Bernd,Köln,50667,41,4100.0,4200.0
3,4,bernd,koeln,50667,x41,4100.0,4200.0
5,6,CHIARA,berlin,10115,35,2950.0,3000.0
8,1,Anna,München,80331,29,3200.0,3300.0


In [ ]:
df_dropna_cols = df.dropna(axis=1)         # Spalten mit NaN löschen
df_dropna_cols

In [ ]:
df_dropna_subset = df.dropna(subset=['Stadt', 'Alter'])  # nur bei bestimmten Spalten löschen
df_dropna_subset

In [ ]:
df_dropna_thresh = df.dropna(thresh=5)     # Mindestanzahl gültiger Werte je Zeile bevor sie gelöscht wird
df_dropna_thresh

In [ ]:
df_fillna_mean = df.copy()
df_fillna_mean[['Einkommen_2023', 'Einkommen_2024']] = df_fillna_mean[
    ['Einkommen_2023', 'Einkommen_2024']
].fillna(df_fillna_mean[['Einkommen_2023', 'Einkommen_2024']].mean(numeric_only=True))
df_fillna_mean          # Mit dem Mittelwert auffüllen

In [ ]:
df_ffill = df.ffill()   # vorwärts auffüllen
df_bfill = df.bfill()   # rückwärts auffüllen
df_ffill

## 2. Duplikate

### 2.1 Duplikate erkennen

In [ ]:
df.duplicated()             # Boolesche Maske

In [ ]:
df.duplicated().sum()       # Anzahl Duplikate

In [ ]:
df[df.duplicated(keep=False)]  # alle Duplikat-Zeilen anzeigen (auch das erste Vorkommen)

### 2.2 Duplikate entfernen

In [ ]:
df_no_dupes = df.drop_duplicates()   # exakte Duplikate entfernen
df_no_dupes

In [ ]:
df_dupes_subset = df.drop_duplicates(subset=['ID'])  # nur anhand bestimmter Spalten
df_dupes_subset

In [ ]:
df_keep_last = df.drop_duplicates(keep='last')  # letztes statt erstes Vorkommen behalten
df_keep_last

## 3. Tippfehler, Schreibweisen und Formate

### 3.1 Text vereinheitlichen

In [ ]:
df_clean = df.copy()
df_clean['Name'] = df_clean['Name'].str.strip()   # Leerzeichen entfernen
df_clean['Name']

In [ ]:
df_clean['Name_lower'] = df_clean['Name'].str.lower()   # klein
df_clean['Name_upper'] = df_clean['Name'].str.upper()   # groß
df_clean['Name_title'] = df_clean['Name'].str.title()   # Titel-Schreibweise
df_clean[['Name', 'Name_lower', 'Name_upper', 'Name_title']]

In [ ]:
# Beispieltext mit Mehrfach-Leerzeichen zum Testen
beispiel = pd.Series(['Anna   Muster', 'Bernd    Beispiel'])
beispiel.str.replace(r'\s+', ' ', regex=True)   # Mehrfach-Leerzeichen reduzieren

In [ ]:
df_clean['Stadt_clean'] = (
    df_clean['Stadt']
    .str.strip()
    .str.lower()
    .str.replace('ä', 'ae')
    .str.replace('ö', 'oe')
    .str.replace('ü', 'ue')
)   # Sonderzeichen/Umlaute vereinheitlichen
df_clean[['Stadt', 'Stadt_clean']]

In [ ]:
mapping_dict = {
    'muenchen': 'München',
    'koeln': 'Köln',
    'berlin': 'Berlin',
    'hamburg': 'Hamburg',
}
df_clean['Stadt_final'] = df_clean['Stadt_clean'].replace(mapping_dict)  # Werte per Dictionary ersetzen
df_clean[['Stadt', 'Stadt_clean', 'Stadt_final']]

In [ ]:
from rapidfuzz import process, fuzz

referenzliste = ['Anna', 'Bernd', 'Chiara', 'Dénis', 'Elke']
wert = 'Denis'

process.extractOne(wert, referenzliste, scorer=fuzz.ratio)  # Fuzzy Matching für Tippfehler

### 3.2 Datentypen & Formate korrigieren

In [ ]:
df_clean['Alter_numeric'] = pd.to_numeric(df_clean['Alter'], errors='coerce')  # Text → Zahl, Fehler → NaN
df_clean[['Alter', 'Alter_numeric']]

In [ ]:
# Beispiel mit Datumsangaben, teils fehlerhaft
datum_beispiel = pd.Series(['2023-01-15', '15.02.2023', 'kein_datum', '2023/03/10'])
pd.to_datetime(datum_beispiel, errors='coerce')   # Text → Datum, Fehler → NaN

In [ ]:
df_clean['PLZ_padded'] = df_clean['PLZ'].astype(str).str.zfill(5)  # führende Nullen erhalten (z. B. PLZ)
df_clean[['PLZ', 'PLZ_padded']]

## 4. Wide & Long Format

Unser Datensatz hat die Einkommensdaten für 2023 und 2024 in zwei separaten Spalten
(`Einkommen_2023`, `Einkommen_2024`) — das ist das **Wide-Format**.
Für viele Analysen (z. B. Zeitverläufe, Gruppierungen) ist das **Long-Format** praktischer.

In [11]:
# Hinweis: ID 1 kommt im Rohdatensatz doppelt vor (siehe Abschnitt 2).
# Wir brauchen eindeutige IDs, deshalb hier vorher bereinigen.
df_wide_beispiel = df[['ID', 'Einkommen_2023', 'Einkommen_2024']].drop_duplicates(subset=['ID'])
df_wide_beispiel

,ID,Einkommen_2023,Einkommen_2024
0,1,3200.0,3300.0
1,2,3200.0,3300.0
2,3,4100.0,4200.0
3,4,4100.0,4200.0
4,5,NaN,3100.0
5,6,2950.0,3000.0
6,7,3800.0,NaN
7,8,3800.0,3900.0
9,9,5000.0,5200.0


In [ ]:
df_long = pd.melt(
    df_wide_beispiel,
    id_vars=['ID'],
    value_vars=['Einkommen_2023', 'Einkommen_2024'],
    var_name='Variable',
    value_name='Wert'
)   # Wide → Long
df_long

In [ ]:
df_wide_zurueck = df_long.pivot(index='ID', columns='Variable', values='Wert')  # Long → Wide (Rückweg)
df_wide_zurueck

*Alternative* für Spalten im Muster `spalte_JAHR` (z. B. `Einkommen_2023`, `Einkommen_2024`): `pd.wide_to_long`. Dafür müssen die Spalten in der Form `stub_suffix` benannt sein — in unserem Fall passt das Muster `Einkommen_2023` / `Einkommen_2024` bereits.

In [ ]:
df_long2 = pd.wide_to_long(
    df_wide_beispiel,
    stubnames='Einkommen',
    i='ID',
    j='Jahr',
    sep='_',
    suffix=r'\d+'
)   # Wide → Long bei Spalten wie "Einkommen_2023", "Einkommen_2024"
df_long2.reset_index()

## ✅ Zusammenfassung

| Schritt | Wichtigste Befehle |
|---|---|
| Fehlende Werte erkennen | `isnull()`, `info()` |
| Fehlende Werte behandeln | `dropna()`, `fillna()`, `ffill()`, `bfill()` |
| Duplikate erkennen | `duplicated()` |
| Duplikate entfernen | `drop_duplicates()` |
| Text vereinheitlichen | `str.strip()`, `str.lower()`, `str.replace()`, `replace()`, `rapidfuzz` |
| Formate korrigieren | `pd.to_numeric()`, `pd.to_datetime()`, `str.zfill()` |
| Wide → Long | `pd.melt()`, `pd.wide_to_long()` |
| Long → Wide | `pivot()` |

Du kannst den `df` oben jederzeit durch deinen **eigenen Datensatz** ersetzen
(z. B. via `pd.read_csv('deine_datei.csv')`) und die gleichen Schritte anwenden.